# Model Step Trace

This notebook loads one trained SNN run, pushes one validation example through the entire model, and records:

- tensor shape after each important step
- a small signal example from that tensor
- basic signal statistics
- a readable step-by-step explanation of what the signal represents


In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path
import copy
import json
import sys

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display
from omegaconf import OmegaConf


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'model_training' / 'trainer.py').exists():
            return path
    raise FileNotFoundError('Could not find repo root containing model_training/trainer.py')


REPO_ROOT = find_repo_root()
MODEL_TRAINING_DIR = REPO_ROOT / 'model_training'

for path in [REPO_ROOT, MODEL_TRAINING_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from trainer import BrainToTextDecoder_Trainer
from helpers.dataset import BrainToTextDataset

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
torch.set_grad_enabled(False)

In [ ]:
RUN_DIR = REPO_ROOT / 'model_training' / 'logs' / 'snn' / '2026-04-04_19-54-38_391644_snn_pid164025'
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / 'best_checkpoint'
SAMPLE_BATCH_INDEX = 0
SAMPLE_IN_BATCH_INDEX = 0
PREVIEW_VALUES = 8

print(f'Run dir: {RUN_DIR}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')

In [ ]:
def load_run_config(run_dir: Path):
    config_path = run_dir / 'configs' / 'trainer_config_resolved.yaml'
    with open(config_path, 'r', encoding='utf-8') as f:
        args = yaml.safe_load(f)
    if 'snn_config' in args and isinstance(args['snn_config'], dict):
        args['snn_config'] = OmegaConf.create(args['snn_config'])
    return args


def override_lagged_lags_from_run_exports(args, run_dir: Path):
    lag_dir = run_dir / 'lag_selections'
    distribution_paths = sorted(lag_dir.glob('block_*_lag_distribution.csv'))
    if not distribution_paths:
        return args

    lag_df = pd.read_csv(distribution_paths[0])
    if 'lag' not in lag_df.columns or lag_df.empty:
        return args

    inferred_lags = [int(v) for v in lag_df['lag'].tolist()]
    snn_core = args['snn_config'].model.setup.snn_core
    configured_lags = [int(v) for v in snn_core.lagged_lags]
    if inferred_lags != configured_lags:
        print(
            'Overriding lagged_lags from run exports:',
            configured_lags,
            '->',
            inferred_lags,
        )
        snn_core.lagged_lags = inferred_lags
    return args


@contextmanager
def _safe_train_batch_index_patch():
    original_fn = BrainToTextDataset.create_batch_index_train

    def _patched(self):
        original_trial_indices = self.trial_indicies
        non_empty = {
            day: info for day, info in self.trial_indicies.items()
            if len(info['trials']) > 0
        }
        if not non_empty:
            return {batch_idx: {} for batch_idx in range(int(self.n_batches or 0))}
        self.trial_indicies = non_empty
        original_n_days = self.n_days
        self.n_days = len(non_empty)
        try:
            return original_fn(self)
        finally:
            self.trial_indicies = original_trial_indices
            self.n_days = original_n_days

    BrainToTextDataset.create_batch_index_train = _patched
    try:
        yield
    finally:
        BrainToTextDataset.create_batch_index_train = original_fn


def build_trainer(run_dir: Path, checkpoint_path: Path | None = None):
    args = copy.deepcopy(load_run_config(run_dir))
    args = override_lagged_lags_from_run_exports(args, run_dir)
    analysis_dir = run_dir / 'notebook_analysis' / 'step_trace'
    analysis_dir.mkdir(parents=True, exist_ok=True)

    args['output_dir'] = str(analysis_dir / 'output')
    args['checkpoint_dir'] = str(analysis_dir / 'checkpoints')
    args['gpu_number'] = '0' if torch.cuda.is_available() else 'cpu'
    args['dataset']['num_dataloader_workers'] = 0
    args['save_best_checkpoint'] = False
    args['save_all_val_steps'] = False
    args['save_final_model'] = False

    with _safe_train_batch_index_patch():
        trainer = BrainToTextDecoder_Trainer(args)

    if checkpoint_path is not None:
        trainer.load_model_checkpoint(str(checkpoint_path))
    trainer.model.eval()
    return trainer


def preview_tensor(x: torch.Tensor, n: int = 8):
    x_cpu = x.detach().cpu()
    if x_cpu.ndim == 0:
        return [float(x_cpu.item())]
    if x_cpu.ndim == 1:
        return x_cpu[:n].tolist()
    if x_cpu.ndim == 2:
        return x_cpu[0, :n].tolist()
    if x_cpu.ndim == 3:
        return x_cpu[0, 0, :n].tolist()
    if x_cpu.ndim == 4:
        return x_cpu[0, 0, :n, 0].tolist()
    flat = x_cpu.reshape(-1)
    return flat[:n].tolist()


def tensor_stats(x: torch.Tensor):
    x_float = x.detach().float().cpu()
    return {
        'shape': tuple(x.shape),
        'dtype': str(x.dtype),
        'min': float(x_float.min().item()),
        'max': float(x_float.max().item()),
        'mean': float(x_float.mean().item()),
        'std': float(x_float.std().item()) if x_float.numel() > 1 else 0.0,
        'preview': preview_tensor(x, PREVIEW_VALUES),
    }


def signal_kind(label: str):
    if 'neuron' in label or 'spike' in label:
        return 'spike-like neural activity'
    if 'logits' in label or 'readout' in label:
        return 'class evidence / decoder output'
    if 'patch' in label:
        return 'patched continuous neural features'
    if 'day transform' in label:
        return 'day-adapted continuous neural features'
    if 'lagged' in label:
        return 'lag-mixed continuous hidden state'
    if 'residual' in label:
        return 'continuous hidden state after skip connection'
    if 'input' in label:
        return 'continuous neural input'
    return 'continuous hidden representation'


def add_trace(trace_rows, label: str, tensor: torch.Tensor, explanation: str):
    stats = tensor_stats(tensor)
    trace_rows.append({
        'step': len(trace_rows),
        'label': label,
        'signal_type': signal_kind(label.lower()),
        'shape': stats['shape'],
        'dtype': stats['dtype'],
        'min': stats['min'],
        'max': stats['max'],
        'mean': stats['mean'],
        'std': stats['std'],
        'preview': stats['preview'],
        'explanation': explanation,
    })


def shift_input_for_lag(x: torch.Tensor, lag: int) -> torch.Tensor:
    if lag == 0:
        return x
    pad = x.new_zeros(lag, x.shape[1], x.shape[2])
    return torch.cat([pad, x[:-lag]], dim=0)


def resolve_trace_dataset(trainer):
    if len(trainer.val_dataset) > 0:
        return trainer.val_dataset, 'val'
    if len(trainer.train_dataset) > 0:
        return trainer.train_dataset, 'train'
    raise ValueError('Both validation and training datasets are empty for this run.')


def resolve_dataset_batch_key(dataset, batch_index: int):
    keys = list(dataset.batch_index.keys())
    if not keys:
        raise ValueError('Selected dataset has no available batches.')
    if batch_index in dataset.batch_index:
        return batch_index
    if 0 <= batch_index < len(keys):
        return keys[batch_index]
    return keys[0]


def load_first_usable_batch(dataset, requested_batch_index: int):
    keys = list(dataset.batch_index.keys())
    preferred_key = resolve_dataset_batch_key(dataset, requested_batch_index)
    candidate_keys = [preferred_key] + [k for k in keys if k != preferred_key]

    last_error = None
    for key in candidate_keys:
        try:
            batch = dataset[key]
        except Exception as exc:
            last_error = exc
            continue
        if len(batch['day_indicies']) == 0:
            continue
        return key, batch

    if last_error is not None:
        raise ValueError(f'Could not load a usable batch from dataset: {last_error}') from last_error
    raise ValueError('Could not find a non-empty batch in the selected dataset.')


def trace_one_example(trainer, batch_index: int = 0, sample_index: int = 0):
    dataset, dataset_name = resolve_trace_dataset(trainer)
    resolved_batch_key, batch = load_first_usable_batch(dataset, batch_index)
    sample_index = min(sample_index, len(batch['day_indicies']) - 1)
    day_idx = batch['day_indicies'][sample_index:sample_index + 1].to(trainer.device)
    n_time_steps = batch['n_time_steps'][sample_index:sample_index + 1].to(trainer.device)

    features = batch['input_features'][sample_index:sample_index + 1].to(trainer.device)
    raw_features = features[:, : int(n_time_steps.item()), :].clone()

    trace_rows = []
    add_trace(trace_rows, 'input_features_raw', raw_features, 'Raw padded dataset input cropped back to the true number of timesteps for one example.')

    transformed_features, transformed_time_steps = trainer.transform_data(features.clone(), n_time_steps.clone(), 'val')
    transformed_features = transformed_features[:, : int(transformed_time_steps.item()), :]
    add_trace(trace_rows, 'input_features_after_val_transform', transformed_features, 'Validation-time signal after smoothing and any validation preprocessing in trainer.transform_data().')

    model = trainer.model
    backbone = model.backbone

    x = model.apply_day_transform(transformed_features, day_idx)
    add_trace(trace_rows, 'after_day_transform', x, 'Day-specific affine transform plus Softsign activation aligns neural input across recording sessions.')

    if model.patch_size > 0:
        x = model.apply_patching(x)
        add_trace(trace_rows, 'after_patching', x, 'Temporal patching groups neighboring timesteps into patch vectors before the SNN backbone.')

    x_tbc = x.permute(1, 0, 2).contiguous()
    add_trace(trace_rows, 'backbone_input_tbc', x_tbc, 'Input is rearranged to time-major format [T, B, C] for the SpikingJelly backbone.')

    z = backbone.input_proj(x_tbc)
    add_trace(trace_rows, 'after_input_projection', z, 'Linear input projection maps each patch vector into the hidden channel dimension used by the SNN blocks.')

    for block_idx, block in enumerate(backbone.snn_core):
        block_in = z
        add_trace(trace_rows, f'block_{block_idx:02d}_input', block_in, f'Input hidden state to residual SNN block {block_idx}.')

        lagged = block.lagged(block_in)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_lagged_mixer', lagged, f'Lagged mixer in block {block_idx} combines delayed versions of channels using its masked lag-specific weights.')

        neuron = block.neuron(lagged)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_neuron', neuron, f'LIF neuron in block {block_idx} turns the mixed continuous signal into spike-like temporal activity.')

        proj = block.proj(neuron)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_linear_proj', proj, f'Per-timestep linear projection remixes hidden channels inside block {block_idx}.')

        bn_in = block.bn_in(proj)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_bn_in', bn_in, f'Reshape before batch normalization in block {block_idx}.')

        bn = block.bn(bn_in)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_batchnorm', bn, f'Batch normalization stabilizes the hidden signal statistics in block {block_idx}.')

        bn_out = block.bn_out(bn)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_bn_out', bn_out, f'Reshape back to [T, B, C] after normalization in block {block_idx}.')

        dropped = block.drop(bn_out)
        add_trace(trace_rows, f'block_{block_idx:02d}_after_dropout', dropped, f'Dropout output for block {block_idx}; in eval mode this should mostly match the pre-dropout tensor.')

        z = dropped + block_in
        add_trace(trace_rows, f'block_{block_idx:02d}_after_residual_add', z, f'Residual connection adds the block input back, producing the final output of block {block_idx}.')

    readout = backbone.readout(z)
    add_trace(trace_rows, 'after_readout', readout, 'Readout layer converts final hidden channels into class logits for each timestep.')

    logits = readout.permute(1, 0, 2).contiguous()
    add_trace(trace_rows, 'final_logits_btc', logits, 'Final output returned by the wrapper model in batch-major format [B, T, classes].')

    meta = {
        'dataset_split': dataset_name,
        'requested_batch_index': int(batch_index),
        'resolved_batch_key': int(resolved_batch_key),
        'resolved_sample_index': int(sample_index),
        'day_index': int(day_idx.item()),
        'n_time_steps': int(n_time_steps.item()),
        'transformed_time_steps': int(transformed_time_steps.item()),
        'block_nums': batch['block_nums'][sample_index].item(),
        'trial_nums': batch['trial_nums'][sample_index].item(),
        'num_trace_steps': len(trace_rows),
    }
    return pd.DataFrame(trace_rows), meta


def render_explanation(trace_df: pd.DataFrame, meta: dict):
    lines = []
    lines.append('## Example metadata')
    for key, value in meta.items():
        lines.append(f'- `{key}`: {value}')
    lines.append('')
    lines.append('## Step-by-step explanation')
    for row in trace_df.itertuples(index=False):
        lines.append(
            f"{row.step}. `{row.label}`: shape `{row.shape}`, signal type `{row.signal_type}`. {row.explanation} "
            f"Preview: `{np.array(row.preview).round(4).tolist()}`."
        )
    return '\n'.join(lines)

In [ ]:
trainer = build_trainer(RUN_DIR, CHECKPOINT_PATH)
trace_df, trace_meta = trace_one_example(
    trainer,
    batch_index=SAMPLE_BATCH_INDEX,
    sample_index=SAMPLE_IN_BATCH_INDEX,
)

display(trace_df[['step', 'label', 'signal_type', 'shape', 'dtype', 'min', 'max', 'mean', 'std', 'preview']])
display(Markdown(render_explanation(trace_df, trace_meta)))

In [ ]:
output_dir = RUN_DIR / 'notebook_analysis' / 'step_trace'
output_dir.mkdir(parents=True, exist_ok=True)

trace_df.to_csv(output_dir / 'model_step_trace.csv', index=False)
with open(output_dir / 'model_step_trace_meta.json', 'w', encoding='utf-8') as f:
    json.dump(trace_meta, f, indent=2)

print(output_dir / 'model_step_trace.csv')
print(output_dir / 'model_step_trace_meta.json')